# Phase 2 — Data Engineering Check
Unpivot the wide sales data into long format and join with calendar + prices.

In [1]:
import os
os.environ["PYSPARK_PYTHON"] = r"D:\Retail Demand Forecasting\.venv\Scripts\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"D:\Retail Demand Forecasting\.venv\Scripts\python.exe"
os.environ["SPARK_LOCAL_DIRS"] = r"D:\spark-temp"
os.environ["HADOOP_HOME"] = r"D:\hadoop"

os.chdir(r"D:\Retail Demand Forecasting")

In [2]:
from pyspark.sql import SparkSession
from retail_demand_forecasting.nodes.data_engineering import unpivot_sales

os.makedirs(r"D:\spark-temp", exist_ok=True)

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("phase2_data_engineering")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .config("spark.local.dir", r"D:\spark-temp")
    .getOrCreate()
)
print(f"Spark {spark.version} ready.")

Spark 4.1.1 ready.


## 1. Load raw datasets

In [3]:
PROJECT_ROOT = r"D:\Retail Demand Forecasting"

sales_train_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(os.path.join(PROJECT_ROOT, "data/01_raw/sales_train_validation.csv"))
)
calendar_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(os.path.join(PROJECT_ROOT, "data/01_raw/calendar.csv"))
)
sell_prices_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(os.path.join(PROJECT_ROOT, "data/01_raw/sell_prices.csv"))
)
print(f"sales_train: {sales_train_raw.count():,} rows, {len(sales_train_raw.columns)} cols")
print(f"calendar:    {calendar_raw.count():,} rows, {len(calendar_raw.columns)} cols")
print(f"sell_prices: {sell_prices_raw.count():,} rows, {len(sell_prices_raw.columns)} cols")

sales_train: 30,490 rows, 1919 cols


calendar:    1,969 rows, 14 cols


sell_prices: 6,841,121 rows, 4 cols


## 2. Run unpivot_sales node

In [4]:
melted_df = unpivot_sales(sales_train_raw, calendar_raw, sell_prices_raw)

## 3. Schema & sample rows

In [5]:
melted_df.printSchema()

root
 |-- day_id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- id: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- dept_id: string (nullable = true)
 |-- cat_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- state_id: string (nullable = true)
 |-- sales: integer (nullable = true)
 |-- sell_price: double (nullable = true)
 |-- wm_yr_wk: integer (nullable = true)
 |-- event_name_1: string (nullable = true)
 |-- event_type_1: string (nullable = true)
 |-- event_name_2: string (nullable = true)
 |-- event_type_2: string (nullable = true)
 |-- snap_CA: integer (nullable = true)
 |-- snap_TX: integer (nullable = true)
 |-- snap_WI: integer (nullable = true)



In [6]:
melted_df.limit(10).toPandas()

,day_id,date,id,item_id,dept_id,cat_id,store_id,state_id,sales,sell_price,wm_yr_wk,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,8,2011-02-05,FOODS_1_177_CA_3_validation,FOODS_1_177,FOODS_1,FOODS,CA_3,CA,1,7.97,11102,None,None,None,None,1,1,1
1,9,2011-02-06,FOODS_1_177_CA_3_validation,FOODS_1_177,FOODS_1,FOODS,CA_3,CA,0,7.97,11102,SuperBowl,Sporting,None,None,1,1,1
2,10,2011-02-07,FOODS_1_177_CA_3_validation,FOODS_1_177,FOODS_1,FOODS,CA_3,CA,0,7.97,11102,None,None,None,None,1,1,0
3,8,2011-02-05,HOBBIES_2_112_TX_2_validation,HOBBIES_2_112,HOBBIES_2,HOBBIES,TX_2,TX,0,NaN,11102,None,None,None,None,1,1,1
4,9,2011-02-06,HOBBIES_2_112_TX_2_validation,HOBBIES_2_112,HOBBIES_2,HOBBIES,TX_2,TX,0,NaN,11102,SuperBowl,Sporting,None,None,1,1,1
5,10,2011-02-07,HOBBIES_2_112_TX_2_validation,HOBBIES_2_112,HOBBIES_2,HOBBIES,TX_2,TX,0,NaN,11102,None,None,None,None,1,1,0
6,1,2011-01-29,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,NaN,11101,None,None,None,None,0,0,0
7,2,2011-01-30,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,NaN,11101,None,None,None,None,0,0,0
8,3,2011-01-31,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,NaN,11101,None,None,None,None,0,0,0
9,4,2011-02-01,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,NaN,11101,None,None,None,None,1,1,0


## 4. Row count

In [7]:
print(f"Total rows: {melted_df.count():,}")

Total rows: 58,327,370


## 5. Save to intermediate parquet

In [8]:
output_path = r"D:\spark-temp\sales_melted.parquet"
melted_sample = melted_df.limit(500000).toPandas()
melted_sample.to_parquet(output_path, index=False)
print(f"Saved {len(melted_sample):,} rows (sample) to {output_path}")

Saved 500,000 rows (sample) to D:\spark-temp\sales_melted.parquet


In [9]:
spark.stop()
print("Phase 2 complete.")

Phase 2 complete.
